# 04 — Baseline ML Pipeline with Degree Features

## Movie Collaboration Network — Blockbuster Prediction

In [ ]:
import pandas as pd
import numpy as np
from neo4j import GraphDatabase

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, f1_score

import warnings
warnings.filterwarnings('ignore')

print('Libraries loaded')

In [ ]:
NEO4J_URI  = 'neo4j://127.0.0.1:7687'
NEO4J_USER = 'neo4j'
NEO4J_PASS = 'Manuvamshi@12'
CLEAN_CSV  = '../data/movies_cleaned.csv'

driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASS))
print('Connected to Neo4j')

## Step 1 — Extract Degree Features from Neo4j

In [ ]:
DEGREE_QUERY = """
MATCH (m:Movie)
OPTIONAL MATCH (m)-[r]-()
WITH m, COUNT(r) AS movie_degree
OPTIONAL MATCH (a:Actor)-[:ACTED_IN]->(m)
WITH m, movie_degree, COUNT(DISTINCT a) AS actor_degree
OPTIONAL MATCH (m)-[:BELONGS_TO]->(g:Genre)
WITH m, movie_degree, actor_degree, COUNT(DISTINCT g) AS genre_degree
OPTIONAL MATCH (d:Director)-[:DIRECTED]->(m)
OPTIONAL MATCH (d)-[:DIRECTED]->(other_m:Movie)
WITH m, movie_degree, actor_degree, genre_degree,
     COUNT(DISTINCT other_m) AS director_total_movies, d
OPTIONAL MATCH (lead:Actor)-[:ACTED_IN]->(m)
OPTIONAL MATCH (lead)-[:ACTED_IN]->(other_m2:Movie)
WITH m, movie_degree, actor_degree, genre_degree, director_total_movies, d,
     COUNT(DISTINCT other_m2) AS actor_total_movies
OPTIONAL MATCH (d)-[:COLLABORATED_WITH]->(collab_a:Actor)
RETURN m.title                       AS node_id,
       movie_degree,
       actor_degree,
       genre_degree,
       director_total_movies,
       actor_total_movies,
       COUNT(DISTINCT collab_a)      AS director_collab_count
"""

with driver.session() as session:
    result = session.run(DEGREE_QUERY)
    degree_df = pd.DataFrame([dict(record) for record in result])

print(f'Extracted degree features for {len(degree_df):,} movie nodes')
print(f'Columns: {list(degree_df.columns)}')
degree_df.head()

In [ ]:
degree_df.describe()

## Step 2 — Build the Feature Matrix

In [ ]:
movies = pd.read_csv(CLEAN_CSV)
print(f'Loaded {len(movies):,} movies from {CLEAN_CSV}')
print(f'Columns: {list(movies.columns)}')
movies.head(3)

In [ ]:
BLOCKBUSTER_THRESHOLD = 100_000_000
movies['is_blockbuster'] = (movies['revenue'] > BLOCKBUSTER_THRESHOLD).astype(int)

print(f'Target distribution:')
print(movies['is_blockbuster'].value_counts())
print(f"\nClass balance: {movies['is_blockbuster'].mean():.1%} blockbusters")

In [ ]:
movies_for_merge = movies.rename(columns={'title': 'node_id'})
matrix = movies_for_merge.merge(degree_df, on='node_id', how='inner')

print(f'Feature matrix shape: {matrix.shape}')
print(f'Movies in CSV but missing from graph: {len(movies_for_merge) - len(matrix):,}')
matrix.head(3)

In [ ]:
numeric_cols = [
    'budget',
    'runtime',
    'release_year',
    'movie_degree',
    'actor_degree',
    'genre_degree',
    'director_total_movies',
    'actor_total_movies',
    'director_collab_count',
]

categorical_cols = [
    'original_language',
]

feature_cols = numeric_cols + categorical_cols

X = matrix[feature_cols].copy()
y = matrix['is_blockbuster'].copy()
node_ids = matrix['node_id'].copy()

print(f'X shape: {X.shape}')
print(f'y shape: {y.shape}')

In [ ]:
print('Missing values per column:')
print(X.isna().sum())
print(f'\nTotal rows with any missing: {X.isna().any(axis=1).sum()}')

mask = ~X.isna().any(axis=1)
X = X[mask].reset_index(drop=True)
y = y[mask].reset_index(drop=True)
node_ids = node_ids[mask].reset_index(drop=True)

print(f'\nAfter dropping NaN rows: X.shape={X.shape}, y.shape={y.shape}')

## Step 3 — Split FIRST, then build the Pipeline

In [ ]:
X_train, X_test, y_train, y_test, ids_train, ids_test = train_test_split(
    X, y, node_ids,
    test_size=0.2,
    stratify=y,
    random_state=42,
)

print(f'Train: {X_train.shape},  positive rate = {y_train.mean():.3f}')
print(f'Test : {X_test.shape},  positive rate = {y_test.mean():.3f}')

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(),                              numeric_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore'),        categorical_cols),
    ]
)

pipe = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier',   RandomForestClassifier(
        n_estimators=200,
        max_depth=None,
        random_state=42,
        n_jobs=-1,
        class_weight='balanced',
    )),
])

print('Pipeline built. Steps:')
for name, step in pipe.named_steps.items():
    print(f'  {name}: {type(step).__name__}')

## Fit and Evaluate

In [ ]:
pipe.fit(X_train, y_train)
print('Pipeline fitted on training data.')

In [ ]:
y_pred = pipe.predict(X_test)

print('=' * 60)
print('BASELINE METRICS')
print('=' * 60)
print(classification_report(y_test, y_pred,
                            target_names=['Not Blockbuster', 'Blockbuster']))

print('Confusion matrix:')
cm = confusion_matrix(y_test, y_pred)
cm_df = pd.DataFrame(
    cm,
    index=['actual: Not BB', 'actual: BB'],
    columns=['pred: Not BB', 'pred: BB'],
)
print(cm_df)

In [ ]:
cv_scores = cross_val_score(pipe, X, y, cv=5, scoring='f1_macro', n_jobs=-1)
print(f'5-fold CV macro-F1: mean = {cv_scores.mean():.4f}  std = {cv_scores.std():.4f}')
print(f'Per-fold scores: {np.round(cv_scores, 4)}')

## Feature Importances

In [ ]:
ohe_cols = pipe.named_steps['preprocessor'].named_transformers_['cat'].get_feature_names_out(categorical_cols)
all_feature_names = list(numeric_cols) + list(ohe_cols)

importances = pipe.named_steps['classifier'].feature_importances_

fi_df = pd.DataFrame({
    'feature':    all_feature_names,
    'importance': importances,
}).sort_values('importance', ascending=False).reset_index(drop=True)

GRAPH_FEATS = {'movie_degree', 'actor_degree', 'genre_degree',
               'director_total_movies', 'actor_total_movies', 'director_collab_count'}
TAB_NUM     = {'budget', 'runtime', 'release_year'}

def tag_family(name):
    if name in GRAPH_FEATS:
        return 'graph (degree)'
    if name in TAB_NUM:
        return 'tabular numeric'
    return 'tabular categorical'

fi_df['family'] = fi_df['feature'].apply(tag_family)
fi_df.head(15)

In [ ]:
summary = fi_df.groupby('family')['importance'].agg(['sum', 'mean', 'count'])
summary['sum_pct'] = (summary['sum'] / summary['sum'].sum() * 100).round(2)
print('Feature importance by family:')
print(summary)
print(f"\nGraph-derived features share: {summary.loc['graph (degree)', 'sum_pct']}%")

## Save the Baseline Matrix for S6

In [ ]:
import os
os.makedirs('../data', exist_ok=True)

baseline_matrix = pd.concat([
    node_ids.rename('node_id'),
    X,
    y.rename('is_blockbuster'),
], axis=1)

out_path = '../data/baseline_feature_matrix.parquet'
baseline_matrix.to_parquet(out_path, index=False)
print(f'Saved baseline matrix to {out_path}')
print(f'Shape: {baseline_matrix.shape}')
print(f'Columns: {list(baseline_matrix.columns)}')

baseline_f1_macro = f1_score(y_test, y_pred, average='macro')
baseline_f1_pos   = f1_score(y_test, y_pred, pos_label=1)

with open('../data/baseline_metrics.txt', 'w') as f:
    f.write(f'baseline_f1_macro={baseline_f1_macro:.4f}\n')
    f.write(f'baseline_f1_blockbuster={baseline_f1_pos:.4f}\n')
    f.write(f'cv_f1_macro_mean={cv_scores.mean():.4f}\n')
    f.write(f'cv_f1_macro_std={cv_scores.std():.4f}\n')

print('\nBaseline metric persisted to ../data/baseline_metrics.txt')
print(f'  test macro-F1       = {baseline_f1_macro:.4f}')
print(f'  test blockbuster F1 = {baseline_f1_pos:.4f}')

In [ ]:
driver.close()
print('Neo4j driver closed.')

---
# Graph-Derived Feature Enrichment (S6)

This section adds **PageRank** and **Louvain community** columns computed in `03_graph_analytics.ipynb`
to the baseline feature matrix, re-trains the same Pipeline, and reports the before-after metric lift.

**Merge key:** `node_id` (= movie title)  
**New numeric feature:** `pagerank`  
**New categorical feature:** `community`

## S6 — Step 1: Load GDS Features

In [ ]:
GDS_PATH = '../data/gds_features.parquet'

gds_features = pd.read_parquet(GDS_PATH)

print(f'Loaded GDS features: {gds_features.shape}')
print(f'Columns: {list(gds_features.columns)}')
print(f'\nPageRank stats:')
print(gds_features['pagerank'].describe().round(4))
print(f'\nCommunity distribution (top 5):')
print(gds_features['community'].value_counts().head())
gds_features.head()

## S6 — Step 2: Merge into Baseline Matrix

In [ ]:
s5_matrix = baseline_matrix.copy()

enriched = pd.merge(
    s5_matrix,
    gds_features,
    how='left',
    on='node_id',
)

enriched['pagerank']  = enriched['pagerank'].fillna(0.0)
enriched['community'] = enriched['community'].fillna(-1).astype(int)

print(f'S5 matrix shape:   {s5_matrix.shape}')
print(f'Enriched shape:    {enriched.shape}')
print(f'Rows matched GDS:  {enriched["pagerank"].gt(0).sum():,}')
print(f'Rows filled with 0 (isolated/missing): {enriched["pagerank"].eq(0).sum():,}')
enriched.head(3)

## S6 — Step 3: Re-build Feature Matrix with New Columns

In [ ]:
numeric_cols_enriched = [
    'budget',
    'runtime',
    'release_year',
    'movie_degree',
    'actor_degree',
    'genre_degree',
    'director_total_movies',
    'actor_total_movies',
    'director_collab_count',
    'pagerank',
]

categorical_cols_enriched = [
    'original_language',
    'community',
]

feature_cols_enriched = numeric_cols_enriched + categorical_cols_enriched

X_enr      = enriched[feature_cols_enriched].copy()
y_enr      = enriched['is_blockbuster'].copy()
node_enr   = enriched['node_id'].copy()

X_enr['community'] = X_enr['community'].astype(str)

print(f'Enriched X shape: {X_enr.shape}')
print(f'New columns added: pagerank, community')

## S6 — Step 4: Split and Re-train the Pipeline

In [ ]:
X_train_e, X_test_e, y_train_e, y_test_e = train_test_split(
    X_enr, y_enr,
    test_size=0.2,
    stratify=y_enr,
    random_state=42,
)

print(f'Train: {X_train_e.shape},  positive rate = {y_train_e.mean():.3f}')
print(f'Test : {X_test_e.shape},  positive rate = {y_test_e.mean():.3f}')

In [ ]:
preprocessor_enr = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(),                       numeric_cols_enriched),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols_enriched),
    ]
)

pipe_enr = Pipeline(steps=[
    ('preprocessor', preprocessor_enr),
    ('classifier',   RandomForestClassifier(
        n_estimators=200,
        max_depth=None,
        random_state=42,
        n_jobs=-1,
        class_weight='balanced',
    )),
])

pipe_enr.fit(X_train_e, y_train_e)
print('Enriched pipeline fitted.')

In [ ]:
y_pred_e = pipe_enr.predict(X_test_e)

print('=' * 60)
print('ENRICHED METRICS (S6: + pagerank + community)')
print('=' * 60)
print(classification_report(y_test_e, y_pred_e,
                            target_names=['Not Blockbuster', 'Blockbuster']))

print('Confusion matrix:')
cm_e = confusion_matrix(y_test_e, y_pred_e)
cm_df_e = pd.DataFrame(
    cm_e,
    index=['actual: Not BB', 'actual: BB'],
    columns=['pred: Not BB', 'pred: BB'],
)
print(cm_df_e)

In [ ]:
cv_scores_e = cross_val_score(pipe_enr, X_enr, y_enr, cv=5, scoring='f1_macro', n_jobs=-1)
print(f'5-fold CV macro-F1: mean = {cv_scores_e.mean():.4f}  std = {cv_scores_e.std():.4f}')
print(f'Per-fold scores: {np.round(cv_scores_e, 4)}')

## S6 — Before vs After Metric Table

In [ ]:
enriched_f1_macro = f1_score(y_test_e, y_pred_e, average='macro')
enriched_f1_pos   = f1_score(y_test_e, y_pred_e, pos_label=1)

comparison = pd.DataFrame({
    'Metric': [
        'F1 macro (test)',
        'F1 blockbuster (test)',
        'CV macro-F1 mean',
        'CV macro-F1 std',
    ],
    'Baseline (S5: degree only)': [
        round(baseline_f1_macro,        4),
        round(baseline_f1_pos,          4),
        round(cv_scores.mean(),         4),
        round(cv_scores.std(),          4),
    ],
    'Enriched (S6: +pagerank +community)': [
        round(enriched_f1_macro,        4),
        round(enriched_f1_pos,          4),
        round(cv_scores_e.mean(),       4),
        round(cv_scores_e.std(),        4),
    ],
})

comparison['Delta'] = (
    comparison['Enriched (S6: +pagerank +community)']
    - comparison['Baseline (S5: degree only)']
).round(4)

print('=' * 70)
print('BEFORE vs AFTER — Graph Analytics Enrichment')
print('=' * 70)
print(comparison.to_string(index=False))
print('=' * 70)

delta_macro = enriched_f1_macro - baseline_f1_macro
if delta_macro > 0.03:
    print(f'\nResult: +{delta_macro:.4f} lift — PageRank and community carry meaningful structural signal.')
elif delta_macro > 0:
    print(f'\nResult: +{delta_macro:.4f} lift — Small but positive. Graph features add marginal signal.')
elif delta_macro == 0:
    print(f'\nResult: No lift — Degree features already captured most local-structure signal.')
else:
    print(f'\nResult: {delta_macro:.4f} delta — Negative. New columns added noise; baseline is the better model.')

## S6 — Feature Importances (Enriched Model)

In [ ]:
ohe_cols_e = pipe_enr.named_steps['preprocessor'].named_transformers_['cat'].get_feature_names_out(categorical_cols_enriched)
all_names_e = list(numeric_cols_enriched) + list(ohe_cols_e)

importances_e = pipe_enr.named_steps['classifier'].feature_importances_

fi_df_e = pd.DataFrame({
    'feature':    all_names_e,
    'importance': importances_e,
}).sort_values('importance', ascending=False).reset_index(drop=True)

GRAPH_FEATS_E = {'movie_degree', 'actor_degree', 'genre_degree',
                 'director_total_movies', 'actor_total_movies',
                 'director_collab_count', 'pagerank'}
TAB_NUM_E     = {'budget', 'runtime', 'release_year'}

def tag_family_e(name):
    if name in GRAPH_FEATS_E:
        return 'graph'
    if name in TAB_NUM_E:
        return 'tabular numeric'
    return 'tabular categorical'

fi_df_e['family'] = fi_df_e['feature'].apply(tag_family_e)

print('Top 15 features by importance (enriched model):')
print(fi_df_e.head(15).to_string(index=False))

In [ ]:
summary_e = fi_df_e.groupby('family')['importance'].agg(['sum', 'mean', 'count'])
summary_e['sum_pct'] = (summary_e['sum'] / summary_e['sum'].sum() * 100).round(2)
print('Feature importance by family (enriched model):')
print(summary_e)
print(f"\nGraph-derived features share: {summary_e.loc['graph', 'sum_pct']}%")

## S6 — Save Enriched Metrics

In [ ]:
with open('../data/enriched_metrics.txt', 'w') as f:
    f.write(f'enriched_f1_macro={enriched_f1_macro:.4f}\n')
    f.write(f'enriched_f1_blockbuster={enriched_f1_pos:.4f}\n')
    f.write(f'cv_f1_macro_mean={cv_scores_e.mean():.4f}\n')
    f.write(f'cv_f1_macro_std={cv_scores_e.std():.4f}\n')
    f.write(f'delta_f1_macro={enriched_f1_macro - baseline_f1_macro:.4f}\n')

print('Enriched metrics saved to ../data/enriched_metrics.txt')
print(f'  Baseline macro-F1:  {baseline_f1_macro:.4f}')
print(f'  Enriched macro-F1:  {enriched_f1_macro:.4f}')
print(f'  Delta:              {enriched_f1_macro - baseline_f1_macro:+.4f}')